# Extended Forecasting Benchmark

Compare seven models at horizons 1–3 from April, August and December 2025. This separate experiment preserves the original March, REER and Suez results. Cohorts and groups use December 2024 information only. Candidate selection uses forecasts made from December 2024 for January–March 2025. LSTM training duration is selected afresh inside each evaluation origin's available history.

## 1. Experiment Settings

Reuse the original series identifiers, metric utility and machine learning candidate dictionaries. Origins denote the final observed month. That month's trade is assumed available. The same targets and series are evaluated by every model; absent monthly trade values remain zero.

In [1]:
from pathlib import Path
import sys
import json
import warnings
import time
import uuid
import gc
from importlib.metadata import version
import numpy as np
import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from chronos import BaseChronosPipeline
from IPython.display import display
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset_loading import (
    DATA_DIR, COUNTRY_PREFIXES, SERIES_KEYS, TARGET, read_notebook_setting,
)
from src.evaluation import evaluate_forecasts
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'extended_benchmark'
COUNTRIES = list(COUNTRY_PREFIXES)
MODELS = ['Naive', 'Seasonal Naive', 'ETS', 'ARIMA', 'XGBoost', 'LSTM', 'Chronos']
ORIGINS = pd.to_datetime(['2025-04-01', '2025-08-01', '2025-12-01'])
HORIZONS = [1, 2, 3]
COHORT_CUTOFF = SELECTION_ORIGIN = pd.Timestamp('2024-12-01')
MAX_HORIZON = max(HORIZONS)
END_MONTH = ORIGINS.max() + pd.DateOffset(months=MAX_HORIZON)
MIN_HISTORY = 36
MAX_ZERO_SHARE = 0.90
TOP_N_PARTNERS = 20
PRODUCT_THRESHOLDS = [200, 150, 100, 50, 1]
SEQUENCE_LENGTH = 12
BATCH_SIZE = 256
RANDOM_SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(2)
CHRONOS_BATCH_SIZE = 32
CHRONOS_ID = 'amazon/chronos-bolt-tiny'
CHRONOS_REVISION = 'f6ff2d2ba9168d498c015bc8dd07e3b395b31b3f'
CHRONOS_CONTEXT_LENGTH = 512
RUN_ID = uuid.uuid4().hex
PROCESSED_FILES = {
    'South Korea': 'korea_trade_monthly.parquet',
    'Slovakia': 'slovakia_trade_monthly.parquet',
    'USA': 'usa_trade_aggregated.parquet'
}
LAG_MONTHS = [1, 2, 3, 6, 12]
ROLLING_WINDOWS = [3, 6, 12]
CHRONOS_TRIAL_SERIES = 8
TRADE_FEATURES = [f'lag_{n}' for n in LAG_MONTHS] + [f'rolling_{stat}_{n}' for stat in [
    'mean',
    'std'
] for n in ROLLING_WINDOWS]
CALENDAR_FEATURES = ['month_of_year', 'month_sin', 'month_cos']
XGB_FEATURES = TRADE_FEATURES + CALENDAR_FEATURES
LSTM_FEATURES = [TARGET] + CALENDAR_FEATURES
XGBOOST_CONFIGS = read_notebook_setting('11_machine_learning_models.ipynb', 'XGBOOST_CONFIGS')
LSTM_CONFIGS = read_notebook_setting('12_deep_model.ipynb', 'LSTM_CONFIGS')
LSTM_MAX_EPOCHS = read_notebook_setting('12_deep_model.ipynb', 'EPOCHS')
LSTM_PATIENCE = read_notebook_setting('12_deep_model.ipynb', 'PATIENCE')
assert SELECTION_ORIGIN + pd.DateOffset(months=3) < ORIGINS.min()
print('Device:', DEVICE, '| PyTorch:', torch.__version__, '| compiled CUDA:', torch.version.cuda)
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0), '| float32')
display(pd.DataFrame([{
    'Origin': o,
    'Horizon': horizon,
    'TargetMonth': o + pd.DateOffset(months=horizon)
} for o in ORIGINS for horizon in HORIZONS]))


Device: cpu | PyTorch: 2.14.1+cpu | compiled CUDA: None


,Origin,Horizon,TargetMonth
0,2025-04-01,1,2025-05-01
1,2025-04-01,2,2025-06-01
2,2025-04-01,3,2025-07-01
3,2025-08-01,1,2025-09-01
4,2025-08-01,2,2025-10-01
5,2025-08-01,3,2025-11-01
6,2025-12-01,1,2026-01-01
7,2025-12-01,2,2026-02-01
8,2025-12-01,3,2026-03-01


## 2. Fixed Cohorts and Groups

Apply the original minimum-history and adaptive partner rules before the 90% zero-share filter, using December 2024 and earlier. Complete each series from its first observation; later missing records become zero and do not cause exclusion. No series cap is applied.

HS chapters use existing HS4 prefixes, without inventing chapter names or changing product classifications. Country-specific size terciles use pre-cutoff mean trade, including zeros. Equal values stay together, so ties can produce unequal or empty size classes. Source-month checks verify country coverage, not extract completeness.

In [ ]:
def select_country_cohort(summary):
    """Freeze partners, eligible series and size groups at the cohort cutoff."""
    eligible = summary.loc[summary.HistoryMonths.ge(MIN_HISTORY)].copy()
    partners = eligible.groupby(
        'Country2',
        observed=True
    ).agg(HistoricalTrade=('HistoricalTrade', 'sum'), Products=('ProductCode', 'nunique')).reset_index()
    partners = partners.sort_values(
        ['HistoricalTrade', 'Country2'],
        ascending=[False, True],
        kind='stable'
    )
    for threshold in PRODUCT_THRESHOLDS:
        chosen = partners.loc[partners.Products.ge(threshold)]
        if len(chosen) >= TOP_N_PARTNERS:
            break
    cohort = eligible.loc[eligible.Country2.isin(chosen.head(TOP_N_PARTNERS).Country2)].copy()
    cohort['ZeroShare'] = 1 - cohort.PositiveMonths / cohort.HistoryMonths
    cohort = cohort.loc[cohort.ZeroShare.le(MAX_ZERO_SHARE)].sort_values(SERIES_KEYS).reset_index(drop=True)
    assert len(cohort) > 0 and (not cohort.duplicated(SERIES_KEYS).any())
    assert cohort.ProductCode.str.fullmatch('\\d{4}').all()
    cohort['HSChapter'] = cohort.ProductCode.str[:2]
    cohort['HistoricalMean'] = cohort.HistoricalTrade / cohort.HistoryMonths
    lower, upper = cohort.HistoricalMean.quantile([1 / 3, 2 / 3]).to_numpy()
    cohort['SizeClass'] = np.select(
        [cohort.HistoricalMean.le(lower), cohort.HistoricalMean.le(upper)],
        ['Small', 'Medium'],
        default='Large'
    )
    cohort['SizeLowerBoundary'], cohort['SizeUpperBoundary'] = (lower, upper)
    cohort['ProductThreshold'] = threshold
    return cohort


### Complete monthly histories

Load the frozen cohorts and check identical target keys at every origin.


In [2]:
def prepare_country_data(country):
    """Load a zero-completed panel without changing the frozen series."""
    path = DATA_DIR / 'processed' / PROCESSED_FILES[country]
    with duckdb.connect() as con:
        con.execute('SET enable_progress_bar=false')
        coverage = con.execute(
            """
            SELECT Month,COUNT(*) AS SourceRows FROM read_parquet(?)
            WHERE Month BETWEEN ? AND ? GROUP BY Month ORDER BY Month
        """,
            [str(path), COHORT_CUTOFF.to_pydatetime(), END_MONTH.to_pydatetime()]
        ).df()
        assert set(pd.date_range(
            COHORT_CUTOFF,
            END_MONTH,
            freq='MS'
        )).issubset(set(pd.to_datetime(coverage.Month)))
        keys = ', '.join(SERIES_KEYS)
        summary = con.execute(
            f"""
            SELECT {keys},MIN(Month) AS FirstMonth,MAX(Month) AS LastObserved,
                SUM(Value) AS HistoricalTrade,COUNT(*) FILTER (WHERE Value>0) AS PositiveMonths,
                date_diff('month',MIN(Month),?::DATE)+1 AS HistoryMonths
            FROM read_parquet(?) WHERE Month<=? GROUP BY {keys}
        """,
            [COHORT_CUTOFF.to_pydatetime(), str(path), COHORT_CUTOFF.to_pydatetime()]
        ).df()
        cohort = select_country_cohort(summary)
        con.register('cohort', cohort)
        panel = con.execute(
            """
            SELECT c.Country1,c.Country2,c.ProductCode,c.TradeFlow,dates.Month,
                   COALESCE(d.Value,0.0)::DOUBLE AS Value,d.Month IS NOT NULL AS SourceObserved
            FROM cohort c CROSS JOIN LATERAL generate_series(
                c.FirstMonth::TIMESTAMP,?::TIMESTAMP,INTERVAL 1 MONTH) dates(Month)
            LEFT JOIN read_parquet(?) d ON d.Country1=c.Country1 AND d.Country2=c.Country2
                AND d.ProductCode=c.ProductCode AND d.TradeFlow=c.TradeFlow AND d.Month=dates.Month
            ORDER BY c.Country1,c.Country2,c.ProductCode,c.TradeFlow,dates.Month
        """,
            [END_MONTH.to_pydatetime(), str(path)]
        ).df()
    panel['Month'] = pd.to_datetime(panel.Month)
    assert not panel.duplicated(SERIES_KEYS + ['Month']).any()
    assert np.isfinite(panel.Value).all() and panel.Value.ge(0).all()
    ordinal = panel.Month.dt.year * 12 + panel.Month.dt.month
    assert ordinal.groupby(
        [panel[k] for k in SERIES_KEYS],
        observed=True
    ).diff().dropna().eq(1).all()
    check = panel.loc[panel.Month.le(COHORT_CUTOFF)].groupby(
        SERIES_KEYS,
        observed=True
    ).Value.agg(['size', 'mean']).reset_index()
    check = cohort.merge(check, on=SERIES_KEYS, validate='one_to_one')
    assert check['size'].eq(check.HistoryMonths).all()
    np.testing.assert_allclose(check['mean'], check.HistoricalMean)
    for origin in ORIGINS:
        for horizon in HORIZONS:
            target = panel.loc[panel.Month.eq(origin + pd.DateOffset(months=horizon))]
            pd.testing.assert_frame_equal(
                target[SERIES_KEYS].reset_index(drop=True),
                cohort[SERIES_KEYS]
            )
    return (panel, cohort, coverage)


## 3. Direct Forecasting Features

At origin t, trade predictors use observations through t. The target is t+h and its calendar is known. A training pair is included only if that target has been observed by the fitting cutoff. Larger horizons therefore have fewer recently labelled examples.

Shifted lag and rolling features follow notebook 07. Their base row is t+1 for every horizon; only the label and target calendar move to t+h. No intermediate future actuals are used as predictors.

In [ ]:
def add_features(panel):
    """Construct shifted historical predictors separately for each trade series."""
    frame = panel.sort_values(SERIES_KEYS + ['Month']).reset_index(drop=True).copy()
    grouped = frame.groupby(SERIES_KEYS, sort=False, observed=True).Value
    for lag in LAG_MONTHS:
        frame[f'lag_{lag}'] = grouped.shift(lag)
    for window in ROLLING_WINDOWS:
        frame[f'rolling_mean_{window}'] = grouped.transform(lambda s: s.shift(1).rolling(window).mean())
        frame[f'rolling_std_{window}'] = grouped.transform(lambda s: s.shift(1).rolling(window).std())
    return frame


def add_calendar_features(frame, months):
    frame = frame.copy()
    frame['month_of_year'] = months.dt.month.to_numpy()
    frame['month_sin'] = np.sin(2 * np.pi * frame.month_of_year / 12)
    frame['month_cos'] = np.cos(2 * np.pi * frame.month_of_year / 12)
    return frame


def build_direct_training_pairs(features, horizon, fit_end):
    """Pair origin-available predictors with labels observed by fit_end."""
    target_values = features.groupby(
        SERIES_KEYS,
        sort=False,
        observed=True
    ).Value.shift(-(horizon - 1))
    months = features.Month + pd.DateOffset(months=horizon - 1)
    use = months.le(fit_end) & target_values.notna()
    pairs = features.loc[use, SERIES_KEYS + ['Month'] + TRADE_FEATURES].copy()
    pairs['TargetMonth'] = months.loc[use]
    pairs['Label'] = target_values.loc[use].to_numpy()
    pairs = add_calendar_features(pairs, pairs.TargetMonth)
    assert pairs.TargetMonth.le(fit_end).all()
    return pairs


def build_forecast_targets(features, cohort, origin, horizon):
    """Use the origin feature row for every horizon and retain keyed actuals."""
    base = features.loc[
        features.Month.eq(origin + pd.DateOffset(months=1)),
        SERIES_KEYS + TRADE_FEATURES
    ]
    target = features.loc[
        features.Month.eq(origin + pd.DateOffset(months=horizon)),
        SERIES_KEYS + ['Value', 'SourceObserved']
    ]
    result = cohort[SERIES_KEYS + ['HSChapter', 'SizeClass', 'HistoricalMean']].merge(
        base,
        on=SERIES_KEYS,
        validate='one_to_one'
    ).merge(target, on=SERIES_KEYS, validate='one_to_one')
    assert len(result) == len(cohort) and result[TRADE_FEATURES].notna().all().all()
    result['Origin'], result['TargetMonth'], result['Horizon'] = (
        origin,
        origin + pd.DateOffset(months=horizon),
        horizon
    )
    return add_calendar_features(result, result.TargetMonth)


### XGBoost configuration selection

Select one configuration per horizon on the earlier holdout.


In [3]:
def build_xgboost_model(params):
    return XGBRegressor(
        **params,
        objective='reg:squarederror',
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )


def select_xgboost_configuration(features, cohort, horizon):
    pairs = build_direct_training_pairs(features, horizon, SELECTION_ORIGIN)
    target = build_forecast_targets(features, cohort, SELECTION_ORIGIN, horizon)
    rows = []
    for name, params in XGBOOST_CONFIGS.items():
        model = build_xgboost_model(params)
        model.fit(pairs[XGB_FEATURES], pairs.Label)
        forecast = np.maximum(model.predict(target[XGB_FEATURES]), 0)
        rows.append({
            'Configuration': name,
            'Horizon': horizon,
            'ValidationMonth': target.TargetMonth.iloc[0],
            'MAE': float(np.abs(target.Value - forecast).mean())
        })
    scores = pd.DataFrame(rows)
    assert np.isfinite(scores.MAE).all()
    display(scores)
    return scores.sort_values('MAE', kind='stable').iloc[0].Configuration


## 4. Direct LSTM and Training Duration

Retain the original twelve-month inputs, log1p transformation, fitting-period scaling, Huber loss, Adam and seeded batches. A separate model predicts each horizon. Architectures are selected once at the December-2024 origin.

At evaluation origin O, select epochs afresh with an internal validation target of O and an internal forecast origin of O−h. Fit through O−h, validate O, then initialise fresh weights and scalers and refit through O for the newly selected duration. The future evaluation target O+h is never used for epochs. Maximum training duration and patience follow notebook 12.

### Direct sequences and architecture

Each training window predicts its horizon-specific target.


In [ ]:
class DirectSequenceDataset(Dataset):

    def __init__(self, frame, horizon, target_month=None):
        self.series, self.index = ([], [])
        for _, series_frame in frame.groupby(SERIES_KEYS, sort=False, observed=True):
            series_frame = series_frame.sort_values('Month')
            values = series_frame[LSTM_FEATURES].to_numpy(dtype=np.float32)
            months = series_frame.Month.to_numpy()
            series_index = len(self.series)
            self.series.append(values)
            for end in range(SEQUENCE_LENGTH - 1, len(series_frame) - horizon):
                target = end + horizon
                if target_month is None or pd.Timestamp(months[target]) == target_month:
                    self.index.append((series_index, end, target))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        series_index, end, target = self.index[i]
        values = self.series[series_index]
        return (
            torch.from_numpy(values[end - SEQUENCE_LENGTH + 1:end + 1].copy()),
            torch.tensor(values[target, 0])
        )


class DirectLSTMForecaster(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            len(LSTM_FEATURES),
            config['hidden_size'],
            config['num_layers'],
            batch_first=True
        )
        self.output = nn.Linear(config['hidden_size'], 1)

    def forward(self, sequences):
        lstm_output, _ = self.lstm(sequences)
        return self.output(lstm_output[:, -1]).squeeze(-1)


### Scaling and epoch selection

Use fitting-period scaling and an internal chronological holdout.


In [ ]:
def scale_lstm_history(panel, fit_end, available_end):
    frame = panel.loc[panel.Month.le(available_end), SERIES_KEYS + ['Month', TARGET]].copy()
    frame = add_calendar_features(frame, frame.Month)
    frame[TARGET] = np.log1p(frame[TARGET])
    scaler = StandardScaler().fit(frame.loc[frame.Month.le(fit_end), LSTM_FEATURES])
    frame[LSTM_FEATURES] = scaler.transform(frame[LSTM_FEATURES])
    return (frame, scaler)


def inverse_lstm_predictions(values, scaler):
    forecast = np.maximum(np.expm1(np.asarray(values) * scaler.scale_[0] + scaler.mean_[0]), 0)
    if not np.isfinite(forecast).all():
        raise ValueError('Non-finite inverse transformation')
    return forecast


def build_lstm_model(config):
    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    return DirectLSTMForecaster(config).to(DEVICE)


def train_lstm_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.HuberLoss()
    for sequences, target_values in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(sequences.to(DEVICE)), target_values.to(DEVICE))
        if not torch.isfinite(loss):
            raise ValueError('Non-finite training loss')
        loss.backward()
        optimizer.step()


### Chronological duration and configuration selection

Select epochs on the internal holdout, and configurations on the earlier holdout.


In [ ]:
def select_lstm_epochs(panel, horizon, config, fit_end, validation_origin):
    """Choose a fresh duration using only the internal chronological holdout."""
    validation_month = validation_origin + pd.DateOffset(months=horizon)
    assert fit_end <= validation_origin
    frame, scaler = scale_lstm_history(panel, fit_end, validation_month)
    train = DirectSequenceDataset(frame.loc[frame.Month.le(fit_end)], horizon)
    validation_dataset = DirectSequenceDataset(frame, horizon, target_month=validation_month)
    assert len(train) > 0 and len(validation_dataset) == panel.loc[panel.Month.eq(validation_month)].shape[0]
    train_loader = DataLoader(
        train,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED)
    )
    val_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
    model = build_lstm_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    best_mae, best_epoch, epochs_without_improvement = (float('inf'), 0, 0)
    history = []
    for epoch in range(1, LSTM_MAX_EPOCHS + 1):
        train_lstm_epoch(model, train_loader, optimizer)
        model.eval()
        predicted = []
        actual = []
        with torch.no_grad():
            for sequences, target_values in val_loader:
                predicted.append(model(sequences.to(DEVICE)).cpu().numpy())
                actual.append(target_values.numpy())
        mae = float(np.mean(np.abs(inverse_lstm_predictions(
            np.concatenate(actual),
            scaler
        ) - inverse_lstm_predictions(np.concatenate(predicted), scaler))))
        history.append({'Epoch': epoch, 'MAE': mae})
        print(f'  epoch {epoch}: validation MAE {mae:,.2f}')
        if mae < best_mae:
            best_mae, best_epoch, epochs_without_improvement = (mae, epoch, 0)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= LSTM_PATIENCE:
            break
    assert best_epoch >= 1
    del model, optimizer
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return (best_epoch, best_mae, pd.DataFrame(history))


def select_lstm_configuration(panel, horizon):
    rows = []
    for name, config in LSTM_CONFIGS.items():
        print(name, 'horizon', horizon)
        epochs, mae, _ = select_lstm_epochs(
            panel,
            horizon,
            config,
            SELECTION_ORIGIN,
            SELECTION_ORIGIN
        )
        rows.append({
            'Configuration': name,
            'Horizon': horizon,
            'MAE': mae,
            'SelectedEpoch': epochs
        })
    scores = pd.DataFrame(rows)
    display(scores)
    return scores.sort_values('MAE', kind='stable').iloc[0].Configuration


### Final LSTM refit

Refit the selected architecture and duration on origin-available history.


In [4]:
def forecast_lstm(panel, target, origin, horizon, config, epochs):
    """Refit on history through the origin and inverse-transform predictions once."""
    frame, scaler = scale_lstm_history(panel, origin, origin)
    training_dataset = DirectSequenceDataset(frame, horizon)
    loader = DataLoader(
        training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED)
    )
    model = build_lstm_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    for epoch in range(epochs):
        train_lstm_epoch(model, loader, optimizer)
        print(f'  refit {epoch + 1}/{epochs}')
    groups = {
        key: series_frame.sort_values('Month') for key,
        series_frame in frame.groupby(SERIES_KEYS, sort=False, observed=True)
    }
    keys = list(target[SERIES_KEYS].itertuples(index=False, name=None))
    expected = pd.date_range(origin - pd.DateOffset(months=SEQUENCE_LENGTH - 1), origin, freq='MS')
    output = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(keys), BATCH_SIZE):
            windows = []
            for key in keys[start:start + BATCH_SIZE]:
                tail = groups[key].tail(SEQUENCE_LENGTH)
                assert np.array_equal(tail.Month.to_numpy(), expected.to_numpy())
                windows.append(tail[LSTM_FEATURES].to_numpy(dtype=np.float32))
            output.append(model(torch.from_numpy(np.stack(windows)).to(DEVICE)).cpu().numpy())
    forecast = inverse_lstm_predictions(np.concatenate(output), scaler)
    del model, optimizer
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return forecast


## 5. Statistical and Pretrained Forecasts

Reuse additive-trend nonseasonal ETS and ARIMA(1,1,1), fitting once per series and origin for all three steps. Handled errors, non-convergence and non-finite forecasts use recorded Naive fallbacks. No series is excluded. Native statistical negative forecasts are retained and counted.

Chronos-Bolt Tiny is a 9M-parameter direct multi-step model. Its pinned November-2024 revision predates selection and evaluation. Overlap with older trade histories in its pretraining collection is unverified; the frozen checkpoint cannot include later 2025 targets via weight updates. Use the median quantile, float32 and at most 512 historical values, without fine-tuning. The first load downloads public weights to the Hugging Face cache.

Sources: [model card](https://huggingface.co/amazon/chronos-bolt-tiny), [revision history](https://huggingface.co/amazon/chronos-bolt-tiny/commits/main), [library](https://github.com/amazon-science/chronos-forecasting).

In [ ]:
def forecast_statistical_models(history, cohort, model_name):
    """Fit each series once and forecast all horizons, recording Naive fallbacks."""
    groups = {
        key: series_frame.sort_values('Month') for key,
        series_frame in history.groupby(SERIES_KEYS, sort=False, observed=True)
    }
    output = []
    fallbacks = []
    reasons = []
    warning_count = 0
    for number, key in enumerate(cohort[SERIES_KEYS].itertuples(index=False, name=None), 1):
        target_values = groups[key].Value.to_numpy(dtype=float)
        fallback = False
        reason = ''
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            try:
                if model_name == 'ARIMA':
                    fitted = ARIMA(
                        target_values,
                        order=(1, 1, 1),
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    ).fit()
                    converged = fitted.mle_retvals.get('converged', True)
                else:
                    fitted = ExponentialSmoothing(
                        target_values,
                        trend='add',
                        seasonal=None,
                        initialization_method='estimated'
                    ).fit()
                    converged = fitted.mle_retvals.get('success', True)
                forecast = np.asarray(fitted.forecast(MAX_HORIZON), dtype=float)
                if not converged or not np.isfinite(forecast).all():
                    raise ValueError('Non-converged or non-finite statistical forecast')
            except (
                ValueError,
                np.linalg.LinAlgError,
                RuntimeError,
                FloatingPointError,
                OverflowError
            ) as exc:
                forecast = np.repeat(target_values[-1], MAX_HORIZON)
                fallback = True
                reason = str(exc)
        warning_count += len(caught)
        output.append(forecast)
        fallbacks.append(fallback)
        reasons.append(reason)
        if number % 1000 == 0:
            print(model_name, number, '/', len(cohort), 'series fitted')
    return (np.stack(output), np.asarray(fallbacks), reasons, warning_count)


### Chronos median forecasts

Use the pinned checkpoint and origin-available contexts without fine-tuning.


In [5]:
def load_chronos():
    pipeline = BaseChronosPipeline.from_pretrained(
        CHRONOS_ID,
        revision=CHRONOS_REVISION,
        device_map=str(DEVICE),
        torch_dtype=torch.float32
    )
    assert pipeline.model.chronos_config.context_length >= CHRONOS_CONTEXT_LENGTH
    return pipeline


def forecast_chronos(pipeline, history, cohort):
    groups = {
        key: series_frame.sort_values('Month').Value.to_numpy(dtype=np.float32) for key,
        series_frame in history.groupby(SERIES_KEYS, sort=False, observed=True)
    }
    keys = list(cohort[SERIES_KEYS].itertuples(index=False, name=None))
    quantiles = np.asarray(pipeline.model.chronos_config.quantiles)
    median = np.flatnonzero(np.isclose(quantiles, 0.5))
    assert len(median) == 1
    predictions = []
    with torch.inference_mode():
        for start in range(0, len(keys), CHRONOS_BATCH_SIZE):
            context = [torch.tensor(groups[k][-CHRONOS_CONTEXT_LENGTH:]) for k in keys[start:start + CHRONOS_BATCH_SIZE]]
            result = pipeline.predict(context, prediction_length=MAX_HORIZON)
            assert result.shape[1] == len(quantiles)
            predictions.append(result[:, int(median[0]), :MAX_HORIZON].cpu().numpy())
    result = np.maximum(np.concatenate(predictions), 0)
    assert result.shape == (len(cohort), MAX_HORIZON) and np.isfinite(result).all()
    return result


## 6. Run and Save Matched Forecasts

Process one country at a time, with no cohort cap. Select configurations once, refit at each origin and select fresh LSTM epochs internally. A small Chronos trial reports inference time before country-wide prediction.

Save completed blocks to a partial Parquet stream. Only a complete seven-model experiment replaces predictions.parquet. model_settings.csv records configuration, epoch selection dates, epochs, device, runtimes, fallback counts and checkpoint provenance. Validation scores remain displayed. Interrupted partial files are not accepted by notebook 17. No feature datasets or model weights are saved.

### Prediction blocks and metadata

Write each matched block with its model settings.


In [ ]:
def save_block(
    writer,
    settings,
    partial_path,
    target,
    country,
    model_name,
    forecast,
    config,
    params=None,
    epochs=None,
    fallback=None,
    reasons=None,
    warnings_count=0,
    elapsed=0,
    epoch_origin=None
):
    """Write one matched block and append settings using the explicitly supplied writer."""
    assert len(forecast) == len(target) and np.isfinite(forecast).all()
    fallback = np.zeros(
        len(target),
        dtype=bool
    ) if fallback is None else np.asarray(fallback, dtype=bool)
    reasons = [''] * len(target) if reasons is None else list(reasons)
    frame = target[SERIES_KEYS + [
        'Origin',
        'TargetMonth',
        'Horizon',
        'Value',
        'SourceObserved',
        'HSChapter',
        'SizeClass',
        'HistoricalMean'
    ]].copy()
    frame['Country'], frame['Model'], frame['Prediction'] = (
        country,
        model_name,
        np.asarray(forecast, dtype=float)
    )
    frame['Fallback'], frame['FallbackReason'] = (fallback, reasons)
    frame['RunID'], frame['CohortCutoff'] = (RUN_ID, COHORT_CUTOFF)
    assert not frame.duplicated(SERIES_KEYS).any()
    arrow = pa.Table.from_pandas(frame, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(partial_path, arrow.schema, compression='zstd')
    try:
        writer.write_table(arrow)
        origin = target.Origin.iloc[0]
        horizon = int(target.Horizon.iloc[0])
        record = {
            'RunID': RUN_ID,
            'Country': country,
            'Model': model_name,
            'Origin': origin,
            'Horizon': horizon,
            'TargetMonth': target.TargetMonth.iloc[0],
            'N Series': len(target),
            'Configuration': config,
            'Parameters': json.dumps(params or {}, sort_keys=True),
            'Epochs': epochs,
            'EpochValidationOrigin': epoch_origin,
            'EpochValidationTarget': origin if model_name == 'LSTM' else pd.NaT,
            'FitThrough': origin,
            'CohortCutoff': COHORT_CUTOFF,
            'SelectionOrigin': SELECTION_ORIGIN,
            'Fallbacks': int(fallback.sum()),
            'NegativeForecasts': int((np.asarray(forecast) < 0).sum()),
            'FitWarnings': warnings_count,
            'ElapsedSeconds': elapsed,
            'ElapsedScope': 'origin_all_horizons' if model_name in [
                'ETS',
                'ARIMA',
                'Chronos'
            ] else 'horizon',
            'Checkpoint': CHRONOS_ID if model_name == 'Chronos' else '',
            'CheckpointRevision': CHRONOS_REVISION if model_name == 'Chronos' else '',
            'PointForecast': 'median' if model_name == 'Chronos' else 'point',
            'Device': str(DEVICE) if model_name in ['LSTM', 'Chronos'] else 'cpu',
            'LibraryVersion': version({
                'XGBoost': 'xgboost',
                'LSTM': 'torch',
                'Chronos': 'chronos-forecasting',
                'ETS': 'statsmodels',
                'ARIMA': 'statsmodels'
            }.get(model_name, 'numpy'))
        }
        settings.append(record)
        score = evaluate_forecasts(
            country,
            model_name,
            target.Value,
            forecast,
            str(target.TargetMonth.iloc[0].date())
        )
        print(f"{country} | {model_name} | {origin:%Y-%m} | h={horizon} | n={len(target):,} | sMAPE={score['sMAPE (%)']:.2f}")
        return writer
    except BaseException:
        # Close partial output even if writing or metadata recording is interrupted.
        writer.close()
        raise


### Models at one forecast origin

Keep the same targets and model execution order.


In [ ]:
def forecast_one_origin(
    country, panel, features, cohort, origin, selected_xgb, selected_lstm,
    pipeline, writer, settings, partial_path,
):
    """Run all models on matched targets in the original execution order."""
    try:
        history = panel.loc[panel.Month.le(origin)]
        targets = {
            horizon: build_forecast_targets(features, cohort, origin, horizon)
            for horizon in HORIZONS
        }
        last = history.loc[history.Month.eq(origin)].set_index(SERIES_KEYS).Value

        # Baselines use the same information set for all three horizons.
        for horizon, target in targets.items():
            keys = pd.MultiIndex.from_frame(target[SERIES_KEYS])
            writer = save_block(
                writer, settings, partial_path, target, country, 'Naive',
                last.reindex(keys).to_numpy(), 'last_value',
            )
            seasonal_month = origin + pd.DateOffset(months=horizon - 12)
            seasonal = history.loc[history.Month.eq(seasonal_month)].set_index(SERIES_KEYS).Value
            writer = save_block(
                writer, settings, partial_path, target, country, 'Seasonal Naive',
                seasonal.reindex(keys).to_numpy(), 'same_month_last_year',
            )

        # Each local statistical fit produces all horizons together.
        for name in ['ETS', 'ARIMA']:
            started = time.perf_counter()
            forecasts, fallbacks, reasons, warn_count = forecast_statistical_models(history, cohort, name)
            elapsed = time.perf_counter() - started
            for horizon, target in targets.items():
                writer = save_block(
                    writer, settings, partial_path, target, country, name,
                    forecasts[:, horizon - 1],
                    'additive_trend' if name == 'ETS' else 'ARIMA(1,1,1)',
                    fallback=fallbacks, reasons=reasons, warnings_count=warn_count, elapsed=elapsed,
                )

        # Direct learned models are refitted separately for each horizon.
        for horizon, target in targets.items():
            name = selected_xgb[horizon]
            started = time.perf_counter()
            train = build_direct_training_pairs(features, horizon, origin)
            model = build_xgboost_model(XGBOOST_CONFIGS[name])
            model.fit(train[XGB_FEATURES], train.Label)
            forecast = np.maximum(model.predict(target[XGB_FEATURES]), 0)
            writer = save_block(
                writer, settings, partial_path, target, country, 'XGBoost',
                forecast, name, XGBOOST_CONFIGS[name], elapsed=time.perf_counter() - started,
            )
            del train, model

            name = selected_lstm[horizon]
            started = time.perf_counter()
            internal_origin = origin - pd.DateOffset(months=horizon)
            epochs, _, epoch_history = select_lstm_epochs(
                panel, horizon, LSTM_CONFIGS[name], internal_origin, internal_origin,
            )
            display(epoch_history)
            forecast = forecast_lstm(panel, target, origin, horizon, LSTM_CONFIGS[name], epochs)
            writer = save_block(
                writer, settings, partial_path, target, country, 'LSTM',
                forecast, name, LSTM_CONFIGS[name], epochs=epochs,
                epoch_origin=internal_origin, elapsed=time.perf_counter() - started,
            )

        started = time.perf_counter()
        forecasts = forecast_chronos(pipeline, history, cohort)
        elapsed = time.perf_counter() - started
        for horizon, target in targets.items():
            writer = save_block(
                writer, settings, partial_path, target, country, 'Chronos',
                forecasts[:, horizon - 1], 'Chronos-Bolt Tiny',
                {'context_length': CHRONOS_CONTEXT_LENGTH, 'batch_size': CHRONOS_BATCH_SIZE},
                elapsed=elapsed,
            )
        return writer
    except BaseException:
        # Propagate failures and interruptions after closing the partial stream.
        if writer is not None:
            writer.close()
        raise


### Country-by-country execution

Select configurations once, then refit at each origin.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
partial_path = OUTPUT_DIR / 'predictions.partial.parquet'
settings_partial = OUTPUT_DIR / 'model_settings.partial.csv'
writer = None
settings = []
cohort_summaries = []
pipeline = load_chronos()
try:
    for country in COUNTRIES:
        print('\nCOUNTRY:', country)
        panel, cohort, coverage = prepare_country_data(country)
        features = add_features(panel)
        cohort_summaries.append({
            'Country': country,
            'N Series': len(cohort),
            **{label: int(cohort.SizeClass.eq(label).sum()) for label in [
                'Small',
                'Medium',
                'Large'
            ]},
            'HS Chapters': cohort.HSChapter.nunique(),
            'SizeLowerBoundary': float(cohort.SizeLowerBoundary.iloc[0]),
            'SizeUpperBoundary': float(cohort.SizeUpperBoundary.iloc[0])
        })
        display(pd.DataFrame(cohort_summaries).tail(1))
        display(coverage)
        trial_cohort = cohort.head(CHRONOS_TRIAL_SERIES)
        trial = panel.loc[panel.Month.le(ORIGINS[0])].merge(
            trial_cohort[SERIES_KEYS],
            on=SERIES_KEYS,
            validate='many_to_one'
        )
        started = time.perf_counter()
        trial_forecast = forecast_chronos(pipeline, trial, trial_cohort)
        print(
            'Chronos trial:',
            len(trial_cohort),
            'series,',
            round(time.perf_counter() - started, 2),
            'seconds,',
            DEVICE
        )
        assert np.isfinite(trial_forecast).all()
        selected_xgb = {horizon: select_xgboost_configuration(
            features,
            cohort,
            horizon
        ) for horizon in HORIZONS}
        selected_lstm = {horizon: select_lstm_configuration(panel, horizon) for horizon in HORIZONS}
        for origin in ORIGINS:
            writer = forecast_one_origin(
                country,
                panel,
                features,
                cohort,
                origin,
                selected_xgb,
                selected_lstm,
                pipeline,
                writer,
                settings,
                partial_path
            )
        del panel, features, cohort
        gc.collect()
finally:
    if writer is not None:
        writer.close()


### Completion checks and final files

Publish final predictions and settings only after the complete experiment.


In [6]:
settings_table = pd.DataFrame(settings)
expected_blocks = len(COUNTRIES) * len(MODELS) * len(ORIGINS) * len(HORIZONS)
assert len(settings_table) == expected_blocks
expected_predictions = (
    sum(row['N Series'] for row in cohort_summaries)
    * len(MODELS) * len(ORIGINS) * len(HORIZONS)
)
prediction_count = int(settings_table['N Series'].sum())
assert prediction_count == expected_predictions == pq.ParquetFile(partial_path).metadata.num_rows
assert not settings_table.duplicated(['Country', 'Model', 'Origin', 'Horizon']).any()
settings_table.to_csv(settings_partial, index=False)
partial_path.replace(OUTPUT_DIR / 'predictions.parquet')
settings_partial.replace(OUTPUT_DIR / 'model_settings.csv')
display(pd.DataFrame(cohort_summaries))
print(
    'Saved',
    prediction_count,
    'predictions and',
    len(settings_table),
    'setting records to',
    OUTPUT_DIR
)



COUNTRY: South Korea


,Country,N Series,Small,Medium,Large,HS Chapters,SizeLowerBoundary,SizeUpperBoundary
0,South Korea,28231,9411,9410,9410,97,16527.777778,248333.333333


,Month,SourceRows
0,2024-12-01,88802
1,2025-01-01,82346
2,2025-02-01,87112
3,2025-03-01,87574
4,2025-04-01,88692
5,2025-05-01,87214
6,2025-06-01,87566
7,2025-07-01,89666
8,2025-08-01,86956
9,2025-09-01,91066


Chronos trial: 8 series, 0.16 seconds, cpu


,Configuration,Horizon,ValidationMonth,MAE
0,XGBoost_1,1,2025-01-01,1.412304e+06
1,XGBoost_2,1,2025-01-01,1.328970e+06
2,XGBoost_3,1,2025-01-01,1.318014e+06


,Configuration,Horizon,ValidationMonth,MAE
0,XGBoost_1,2,2025-02-01,1.499662e+06
1,XGBoost_2,2,2025-02-01,1.373251e+06
2,XGBoost_3,2,2025-02-01,1.361555e+06


... (86 outputs left out)

,Epoch,MAE
0,1,4.050589e+06
1,2,3.765302e+06
2,3,3.979864e+06
3,4,3.769402e+06
4,5,3.343569e+06
5,6,3.645995e+06
6,7,3.497937e+06
7,8,3.671720e+06
8,9,3.417286e+06
9,10,3.811677e+06


  refit 1/5
  refit 2/5
  refit 3/5
  refit 4/5
  refit 5/5
USA | LSTM | 2025-12 | h=3 | n=39,739 | sMAPE=99.02
USA | Chronos | 2025-12 | h=1 | n=39,739 | sMAPE=83.58
USA | Chronos | 2025-12 | h=2 | n=39,739 | sMAPE=85.65
USA | Chronos | 2025-12 | h=3 | n=39,739 | sMAPE=84.69


,Country,N Series,Small,Medium,Large,HS Chapters,SizeLowerBoundary,SizeUpperBoundary
0,South Korea,28231,9411,9410,9410,97,16527.777778,248333.333333
1,Slovakia,31328,10443,10442,10443,97,2954.996532,45317.382164
2,USA,39739,13247,13246,13246,98,53976.947368,786721.718750


Saved 6255774 predictions and 189 setting records to c:\Transfer\InterdisciplinaryProject\results\extended_benchmark


## Summary

The experiment retains fixed series and HS, flow and size groups across origins, horizons and models. Initial configuration selection precedes evaluation; every LSTM refit selects its own duration internally. The November-2024 Chronos checkpoint receives only origin-available history.

Keyed predictions support detailed comparisons without retraining. The predictions and settings files share a run identifier. A complete run replaces only this extension's final files; partial files are not evaluation inputs. Notebook 17 validates matching before calculating origin-specific metrics and figures.